In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))

/home/gtamo/MS_ML


In [22]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
# from tdc.multi_pred import DTI

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Imports

In [3]:
## params
active_c = '#008bfb' # "#3B85C1"
silent_c = '#ff0051'

# data
RAW_PROTEOMICS_PATH   = 'data/MS/20260424_Proteomics_Database_CSV_Export.csv' # df_raw
CLEAN_PROTEOMICS_PATH = 'data/MS/20260429_CDD_MS_SilentActive.csv' # silent vs active
CHEMLIB_PATH          = 'data/chemical_libs/20260430_SERAC_lib.csv' # smiles + compound
OT_ROOT               = 'data/external/opentarget'
PHARMA_PATENT_CSV     = 'data/patent/20260512_pharma_sm.csv' # pharma targets of interest
PX_SCREEN_LIB         = 'data/MS/20260513_CDD_FBXO31_PxScreen_Source.csv'
# output
OT_CACHE              = 'output/MS/opentargets_target_disease.parquet'
GENE_SAR_OUT          = 'output/MS/20260509_geneSAR_R2_full_genome.csv' # R2 per gene
MCS_CSV               = 'output/MS/20260505_target_final_mcs.csv' # MCS enrichment
ML_MODEL_OUTPUT       = 'output/ML/trained_models/20260513' # dump for trained ML models
# dropbox/system
PATENTS_RAW           = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/5_Literature and Patents/0_Companies/Pharma_SmallMolecule_Patents_MASTER.xlsx'
DROPBOX_PROT          = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/12_Proteomics/5_Inventory/CDDVault/Proteomics/GiorgioTamo/'
DROPBOX_ML            = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/'
DOWNLOADS             = '/mnt/c/Users/gtamo/Downloads/'
ENAMINE_20260513      = DROPBOX_ML+'virtual libraries/enum_NAr-pyrimidine_Enamine 4.sdf'

# misc
CM2RM                 = ['SRB-0005653']
FEATURES_TYPE         = 'prevalence' # 'autoresearch' # 


### 20260528 - Prioritizing virtual & library compounds vs PCSK9
We will use the 20260513 trained models to proritize compounds coming from a virtual enumeration and unscreen compounds from our library

In [4]:
## get enamine virtual enum - convert sdf to smiles and extract relevant fields/columns
enum_df = rdkit_tools.get_smiles_df_from_enum(ENAMINE_20260513)
enum_df['compound']  = enum_df['R1_Code'] + '_' + enum_df['R2_Code']
enum_df['filename'] = Path(ENAMINE_20260513).stem
enum_df['smiles'] = enum_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum_df.head(1)

100%|██████████| 9441/9441 [00:03<00:00, 2860.29it/s]


,smiles,R1_Code,R2_Code,compound,filename
0,CCC(CC)(CC)CNc1nccc(-c2ccc3c(c2)[C@@H]2CNC(=O)...,EN300-106990,EN300-53215384,EN300-106990_EN300-53215384,enum_NAr-pyrimidine_Enamine 4


In [5]:
## get unscreened library:
lib = pd.read_csv(PX_SCREEN_LIB).rename(columns={'Molecule Name':'compound','SMILES':'smiles','Lib ID':'lib_id','Px_screened_source':'source'})
lib['smiles'] = lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
print(lib['Px_screened_anywhere'].unique())
lib['Px_sreened'] = 0
lib.loc[lib['Px_screened_anywhere']=='yes','Px_sreened'] = 1
lib = lib[['compound','smiles','lib_id','source','Px_sreened']]

# select uncreened compounds from enamine 4 lib id
# lib = lib[ (lib['lib_id']=='Enamine 4') & (lib['Px_sreened']==0) ]
# print(lib.shape)
# lib.head(1)

# get all the screened compounds:
Px_screened_smiles = list(lib[lib['Px_sreened']==1]['smiles'])
len(Px_screened_smiles)

  0%|          | 0/6444 [00:00<?, ?it/s]

100%|██████████| 6444/6444 [00:01<00:00, 3444.80it/s]

<ArrowStringArray>
[nan, 'yes']
Length: 2, dtype: str


5097

In [6]:
## remove anything that could have been screened from the enumeration:
print(f"before {enum_df.shape}")
enum_df = enum_df[~enum_df['smiles'].isin(Px_screened_smiles)]
print(f"after {enum_df.shape}")

before (9441, 5)
after (9181, 5)


In [26]:
## prioritize compounds:
## Implementation now lives in Scripts/ML_Reg.py (ML_Reg.MLReg_prioritize_compounds).
## The function loads a joblib bundle, computes H236 features on `ori_data`, and
## returns the top-N most-active compounds. See its docstring for details.
model_path = 'output/ML/trained_models/20260513/PCSK9_RF_H236.joblib'

pred_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum_df,
    model=model_path,
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
)


> PCSK9  R²=0.124  features=H236  (2589 cols)  predicted 9,181 unique / 9,181 rows; top 300


In [32]:
test_outpath = DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf'
pred_df

## Sanity check: SMILES intersection between the written SDF and pred_df
from rdkit import Chem

# 1) Read back what we wrote
out_df = rdkit_tools.get_smiles_df_from_enum(test_outpath)

# 2) Re-canonicalize both sides defensively — different code paths
#    (MolToSmiles in get_smiles_df_from_enum vs convert_smiles_to_canonical
#    upstream of pred_df) usually agree, but a one-pass re-canonicalisation
#    eliminates any kekulization / stereo-perception drift.
def _canon(smi):
    if not isinstance(smi, str) or not smi:
        return None
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else None

out_smis  = set(out_df['smiles'].map(_canon).dropna())
pred_smis = set(pred_df['smiles'].map(_canon).dropna())

inter      = out_smis & pred_smis
only_in_out  = out_smis - pred_smis
only_in_pred = pred_smis - out_smis

print(f'SDF written         : {len(out_df):,} rows  ({len(out_smis):,} unique canonical SMILES)')
print(f'pred_df             : {len(pred_df):,} rows ({len(pred_smis):,} unique canonical SMILES)')
print(f'∩ (intersection)    : {len(inter):,}')
print(f'in SDF, NOT in pred : {len(only_in_out):,}')
print(f'in pred, NOT in SDF : {len(only_in_pred):,}')




SDF written         : 300 rows  (300 unique canonical SMILES)
pred_df             : 300 rows (300 unique canonical SMILES)
∩ (intersection)    : 300
in SDF, NOT in pred : 0
in pred, NOT in SDF : 0


In [35]:
## write sdf in dropbox folder:
if pred_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred_df,
        source_dir=DROPBOX_ML+'virtual libraries', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf',
        pred_col='predicted_label',
)

#### Prioritize 150 compounds from 114 virtual enumeration

In [ ]:
## get 114 virtual enumerations:
enumpath = 'data/enumeration/20260427/'
enum_fs = glob(enumpath+'*')

# get a unified file for prediction:
# combined_sdf, n_parts = rdkit_tools.combine_sdfs(enumpath, '20260518_114K_enum.sdf')
enum114_df = rdkit_tools.get_smiles_df_from_enum_dir(enumpath, v=True)
print(f'> parsed {enum114_df["filename"].nunique()} SDFs, {len(enum114_df):,} rows total')


# get smiles from this:
enum114_df['compound']  = enum114_df['R1_Code'] + '_' + enum114_df['R2_Code']
enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum114_df = enum114_df.drop_duplicates('smiles').reset_index(drop=True)

## remove compounds already present in other sets:
cm2rm = list(enum_df['smiles']) + Px_screened_smiles
print(f"before {enum114_df.shape}")
enum114_df = enum114_df[~enum114_df['smiles'].isin(cm2rm)]
print(f"before {enum114_df.shape}")

# get smiles from this:
enum114_df['compound']  = enum114_df['R1_Code'] + '_' + enum114_df['R2_Code']
enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum114_df = enum114_df.drop_duplicates('smiles').reset_index(drop=True)

## remove compounds already present in other sets:
cm2rm = list(enum_df['smiles']) + Px_screened_smiles
print(f"before {enum114_df.shape}")
enum114_df = enum114_df[~enum114_df['smiles'].isin(cm2rm)]
print(f"before {enum114_df.shape}")


SDF dir:   0%|          | 0/18 [00:00<?, ?file/s]

SDF dir: 100%|██████████| 18/18 [01:40<00:00,  5.59s/file]

> parsed 18 SDFs, 112,078 rows total


In [53]:
## prioritize compounds:
pred114_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum114_df,
    model='output/ML/trained_models/20260513/PCSK9_RF_H236.joblib',
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
    verbose=True,
)

> loading bundle: gene=PCSK9  R²=0.124  features=H236  (2589 cols)
> featurising 96,357 unique compounds via compute_H236_features...


predicting: 100%|██████████| 21/21 [00:05<00:00,  3.61chunk/s]


> PCSK9  R²=0.124  features=H236  (2589 cols)  predicted 96,357 unique / 96,357 rows; top 300


In [56]:
## write sdf in dropbox folder:
if pred114_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred114_df,
        source_dir='data/enumeration/20260427/', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_Enamine_114k_pred.sdf',
        pred_col='predicted_label',
)